<a href="https://colab.research.google.com/github/FerneyEcheverri/Actividad_3/blob/master/Simulacion_Yathzze_Montecarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  ✅ El sistema corresponde a una aplicación interactiva del juego Yahtzee implementada en Python mediante el uso de ipywidgets, basada en un ejemplo entregado por el docente a la cual se le anexó una interacción gráfica dentro del entorno colab. El sistema permite partidas entre dos jugadores, con seguimiento en tiempo real de estadísticas, probabilidades y evolución de puntajes, además cuenta con las siguientes características:
* Variables Aleatorias: 5 dados de 6 caras cada uno.
* Distribución: Uniforme discreta P(X=k) = 1/6 para k ∈ {1,2,3,4,5,6}
* Restricciones: Máximo 3 lanzamientos por turno, posibilidad de bloquear dados

In [2]:
import ipywidgets as widgets # Importa la librería ipywidgets para crear elementos interactivos de la interfaz de usuario.
from IPython.display import display, clear_output # Importa funciones para mostrar y limpiar la salida en Jupyter.
import random # Importa el módulo random para generar números aleatorios (lanzamiento de dados).
from collections import Counter # Importa Counter para contar la frecuencia de elementos en una lista (caras de los dados).
import matplotlib.pyplot as plt # Importa Matplotlib para crear gráficos y visualizaciones.

clear_output(wait=True) # Limpia la salida anterior de la celda.

# ===============================
# Caras visuales de los dados
# ===============================
# Diccionario que mapea el valor numérico de un dado a su representación Unicode.
CARAS_DADOS = {
    1: "⚀", 2: "⚁", 3: "⚂",
    4: "⚃", 5: "⚄", 6: "⚅"
}

# ===============================
# Variables principales del juego
# ===============================
# Inicialización de las variables globales que controlan el estado del juego.
jugador_actual = 1 # Indica qué jugador tiene el turno (1 o 2).
ronda_general = 1 # Contador de la ronda actual del juego.
max_rondas = 13 # Número máximo de rondas a jugar.

dados_actuales = [1, 1, 1, 1, 1] # Lista que guarda el valor de los 5 dados en el turno actual.
tiros_restantes = 3 # Número de lanzamientos de dados restantes en el turno actual.

puntajes = { # Diccionario para almacenar los puntajes de cada jugador por categoría.
    1: {}, # Puntajes del jugador 1.
    2: {}  # Puntajes del jugador 2.
}

historial_jugador1 = [] # Guarda los puntajes obtenidos por el jugador 1 en cada turno.
historial_jugador2 = [] # Guarda los puntajes obtenidos por el jugador 2 en cada turno.

turnos = [] # Lista para almacenar el número de turno para el gráfico de evolución de puntajes.
puntos_j1_grafica = [] # Lista para almacenar los puntajes acumulados del jugador 1 para el gráfico.
puntos_j2_grafica = [] # Lista para almacenar los puntajes acumulados del jugador 2 para el gráfico.

conteo_caras = Counter() # Contador para llevar el registro de cuántas veces ha salido cada cara del dado en todo el juego.

# Conteo real de combinaciones obtenidas durante el juego para estadísticas.
conteo_combinaciones = Counter() # Guarda la frecuencia de las combinaciones (Yahtzee, Póker, etc.) obtenidas.
total_muestras_combinaciones = 0 # Contador del total de lanzamientos de dados para calcular probabilidades reales.

# Lista de todas las categorías de puntuación en el juego Yahtzee.
categorias = [
    "Unos", "Dos", "Tres", "Cuatro", "Cinco", "Seises", # Categorías superiores (suma de valores de dados).
    "Tres iguales", "Cuatro iguales", "Casa llena", # Categorías inferiores basadas en combinaciones.
    "Pequeño recto", "Recto grande", "Oportunidad", "Yahtzee"
]

# Lista de combinaciones de dados usadas para las estadísticas de probabilidad.
combinaciones_estadisticas = [
    "Yahtzee / 5 iguales",
    "Póker / 4 iguales",
    "Full House / Casa llena",
    "Escalera mayor",
    "Escalera menor",
    "Tres iguales",
    "Doble pareja",
    "Una pareja",
    "Sin combinación especial"
]

# ===============================
# Funciones de puntuación del juego
# ===============================
# Función para verificar si hay una secuencia (escalera) en los dados.
def tiene_secuencia(dados, largo):
    unicos = sorted(set(dados)) # Obtiene valores únicos y los ordena.
    contador = 1 # Inicializa el contador de secuencia.

    # Itera sobre los valores únicos para encontrar secuencias.
    for i in range(1, len(unicos)):
        if unicos[i] == unicos[i - 1] + 1: # Si el valor actual es consecutivo al anterior.
            contador += 1 # Incrementa el contador de secuencia.
            if contador >= largo: # Si la secuencia alcanza el largo deseado.
                return True # Retorna True.
        else:
            contador = 1 # Reinicia el contador si la secuencia se rompe.

    return False # Si no se encuentra la secuencia, retorna False.


# Función para calcular el puntaje de una categoría específica con un set de dados.
def calcular_categoria(dados, categoria):
    conteo = Counter(dados) # Cuenta la frecuencia de cada cara en los dados.
    valores = list(conteo.values()) # Obtiene los conteos de frecuencia (e.g., [3, 2] para Casa llena).
    total = sum(dados) # Suma total de los valores de los dados (usado en varias categorías).

    # Lógica de puntuación para cada categoría.
    if categoria == "Unos":
        return conteo.get(1, 0) * 1 # Suma de todos los unos.
    elif categoria == "Dos":
        return conteo.get(2, 0) * 2 # Suma de todos los dos.
    elif categoria == "Tres":
        return conteo.get(3, 0) * 3 # Suma de todos los tres.
    elif categoria == "Cuatro":
        return conteo.get(4, 0) * 4 # Suma de todos los cuatro.
    elif categoria == "Cinco":
        return conteo.get(5, 0) * 5 # Suma de todos los cinco.
    elif categoria == "Seises":
        return conteo.get(6, 0) * 6 # Suma de todos los seises.
    elif categoria == "Tres iguales":
        return total if any(v >= 3 for v in valores) else 0 # Si hay al menos tres dados iguales, suma todos los dados.
    elif categoria == "Cuatro iguales":
        return total if any(v >= 4 for v in valores) else 0 # Si hay al menos cuatro dados iguales, suma todos los dados.
    elif categoria == "Casa llena":
        return 25 if 3 in valores and 2 in valores else 0 # Si hay tres de un tipo y dos de otro, 25 puntos.
    elif categoria == "Pequeño recto":
        return 30 if tiene_secuencia(dados, 4) else 0 # Si hay una secuencia de 4 dados, 30 puntos.
    elif categoria == "Recto grande":
        return 40 if tiene_secuencia(dados, 5) else 0 # Si hay una secuencia de 5 dados, 40 puntos.
    elif categoria == "Oportunidad":
        return total # Suma total de todos los dados.
    elif categoria == "Yahtzee":
        return 50 if 5 in valores else 0 # Si los 5 dados son iguales, 50 puntos.

    return 0 # Si la categoría no coincide, retorna 0.


# Función que determina la mejor categoría disponible para puntuar con los dados actuales para un jugador.
def mejor_categoria(dados, jugador):
    mejor_cat = None # Variable para almacenar el nombre de la mejor categoría.
    mejor_puntaje = -1 # Variable para almacenar el puntaje de la mejor categoría.

    # Itera sobre todas las categorías posibles.
    for categoria in categorias:
        # Solo considera categorías que el jugador aún no ha usado.
        if categoria not in puntajes[jugador]:
            puntos = calcular_categoria(dados, categoria) # Calcula el puntaje para la categoría actual.

            # Si el puntaje actual es mayor que el mejor encontrado hasta ahora.
            if puntos > mejor_puntaje:
                mejor_puntaje = puntos # Actualiza el mejor puntaje.
                mejor_cat = categoria # Actualiza la mejor categoría.

    return mejor_cat, mejor_puntaje # Retorna la mejor categoría y su puntaje.


# Función para calcular el puntaje total acumulado de un jugador.
def total_jugador(jugador):
    return sum(puntajes[jugador].values()) # Suma todos los puntajes guardados para el jugador.


# Función para calcular la prima de la sección superior del marcador.
def calcular_prima(jugador):
    suma_superior = 0 # Inicializa la suma de la sección superior.

    # Suma los puntajes de las categorías superiores.
    for categoria in ["Unos", "Dos", "Tres", "Cuatro", "Cinco", "Seises"]:
        suma_superior += puntajes[jugador].get(categoria, 0) # Usa .get para manejar categorías no puntuadas aún.

    return 35 if suma_superior >= 63 else 0 # Otorga 35 puntos de prima si la suma es 63 o más.


# ===============================
# Clasificar combinación real de dados para estadísticas
# ===============================
# Función para identificar la combinación específica de los dados lanzados.
def calcular_combinacion_actual(dados):
    conteo = Counter(dados) # Cuenta la frecuencia de cada cara.
    valores = list(conteo.values()) # Obtiene los conteos de las caras.

    # Lógica para identificar las combinaciones.
    if 5 in valores:
        return "Yahtzee / 5 iguales"
    elif 4 in valores:
        return "Póker / 4 iguales"
    elif 3 in valores and 2 in valores:
        return "Full House / Casa llena"
    elif tiene_secuencia(dados, 5):
        return "Escalera mayor"
    elif tiene_secuencia(dados, 4):
        return "Escalera menor"
    elif 3 in valores:
        return "Tres iguales"
    elif valores.count(2) == 2: # Verifica si hay dos pares (e.g., dos 2s y dos 4s).
        return "Doble pareja"
    elif 2 in valores:
        return "Una pareja"
    else:
        return "Sin combinación especial"


# ===============================
# Simulación Montecarlo
# ===============================
# Función para simular lanzamientos de dados y evaluar posibles puntajes.
def simulacion_montecarlo(cantidad=1000):
    resultados = [] # Lista para almacenar los mejores puntajes obtenidos en cada simulación.
    caras_simuladas = Counter() # Contador para la frecuencia de caras en la simulación.

    for i in range(cantidad): # Realiza la cantidad de simulaciones especificada.
        dados_simulados = [] # Dados para cada simulación.

        for j in range(5):
            numero = random.randint(1, 6) # Lanza un dado aleatoriamente.
            dados_simulados.append(numero)
            caras_simuladas[numero] += 1 # Registra la cara simulada.

        # Calcula el mejor puntaje posible para los dados simulados.
        mejor_puntaje = max(calcular_categoria(dados_simulados, cat) for cat in categorias)
        resultados.append(mejor_puntaje)

    return resultados, caras_simuladas # Retorna los resultados de puntajes y el conteo de caras.


# ===============================
# Salidas visuales (widgets de IPython)
# ===============================
# Se crean objetos Output de ipywidgets para mostrar información en diferentes secciones de la interfaz.
turn_output = widgets.Output() # Muestra información del turno (ronda, jugador, tiros restantes).
dice_output = widgets.Output() # Muestra el estado actual de los dados y la mejor categoría.
score_output = widgets.Output() # Muestra la tabla de puntuación.
result_output = widgets.Output() # Muestra mensajes de resultado (ej. "Juego terminado").
stats_output = widgets.Output() # Muestra estadísticas del juego.
probabilidades_output = widgets.Output() # Muestra las probabilidades reales de combinaciones.
grafico_output = widgets.Output() # Muestra los gráficos generados.

# ===============================
# Dados visuales (widgets)
# ===============================
dice_labels = [] # Lista de labels para mostrar las caras de los dados.

# Crea 5 labels, uno para cada dado, usando las caras Unicode.
for i in range(5):
    label = widgets.Label(value=CARAS_DADOS[dados_actuales[i]])
    label.layout = widgets.Layout(width="70px") # Define el ancho del label.
    label.style.font_size = "50px" # Define el tamaño de la fuente.
    dice_labels.append(label)

# Crea 5 checkboxes para bloquear dados individuales.
dice_checkboxes = [
    widgets.Checkbox(value=False, description=f"Bloquear {i + 1}")
    for i in range(5)
]

# Agrupa cada label de dado con su checkbox correspondiente en un VBox (vertical).
contenedores_dados = [
    widgets.VBox([dice_labels[i], dice_checkboxes[i]])
    for i in range(5)
]

dice_container = widgets.HBox(contenedores_dados) # Agrupa todos los contenedores de dados en un HBox (horizontal).

# ===============================
# Botones (widgets)
# ===============================
roll_button = widgets.Button(
    description="Lanzar dados", # Texto del botón.
    button_style="primary" # Estilo visual.
)

save_button = widgets.Button(
    description="Guardar puntaje",
    button_style="success",
    disabled=True # Inicialmente deshabilitado hasta que se lancen los dados.
)

grafico_button = widgets.Button(
    description="Mostrar gráficos",
    button_style="info"
)

# ===============================
# Funciones para actualizar la interfaz visual
# ===============================
# Actualiza la representación visual de los dados en la interfaz.
def actualizar_dados():
    for i in range(5):
        dice_labels[i].value = CARAS_DADOS[dados_actuales[i]] # Asigna la cara Unicode correspondiente al valor del dado.


# Muestra la información del turno actual.
def mostrar_turno():
    with turn_output: # Dirige la salida a la sección 'turn_output'.
        clear_output() # Limpia la salida anterior.
        print(f"Ronda {ronda_general} de {max_rondas}")
        print(f"Turno del Jugador {jugador_actual}")
        print(f"Tiros restantes: {tiros_restantes}")


# Muestra la tabla de puntuación completa de ambos jugadores.
def mostrar_tabla():
    with score_output: # Dirige la salida a la sección 'score_output'.
        clear_output() # Limpia la salida anterior.

        print("TABLA DE PUNTUACIÓN")
        print("-" * 58)
        print(f"{'Categoría':24} {'Jugador 1':14} {'Jugador 2':14}")
        print("-" * 58)

        # Imprime las categorías superiores.
        for categoria in categorias[:6]:
            j1 = puntajes[1].get(categoria, "") # Obtiene el puntaje o una cadena vacía si no existe.
            j2 = puntajes[2].get(categoria, "")
            print(f"{categoria:24} {str(j1):14} {str(j2):14}")

        # Calcula y muestra las sumas de las categorías superiores.
        suma1 = sum(puntajes[1].get(c, 0) for c in categorias[:6])
        suma2 = sum(puntajes[2].get(c, 0) for c in categorias[:6])

        print("-" * 58)
        print(f"{'Suma superior':24} {suma1:<14} {suma2:<14}")
        print(f"{'Prima':24} {calcular_prima(1):<14} {calcular_prima(2):<14}") # Muestra la prima.
        print("-" * 58)

        # Imprime las categorías inferiores.
        for categoria in categorias[6:]:
            j1 = puntajes[1].get(categoria, "")
            j2 = puntajes[2].get(categoria, "")
            print(f"{categoria:24} {str(j1):14} {str(j2):14}")

        # Calcula y muestra los puntajes totales finales (incluyendo prima).
        total1 = total_jugador(1) + calcular_prima(1)
        total2 = total_jugador(2) + calcular_prima(2)

        print("-" * 58)
        print(f"{'Puntuación total':24} {total1:<14} {total2:<14}")


# Muestra las estadísticas generales del juego.
def mostrar_estadisticas():
    # Calcula promedios y mejores turnos.
    promedio1 = sum(historial_jugador1) / len(historial_jugador1) if historial_jugador1 else 0
    promedio2 = sum(historial_jugador2) / len(historial_jugador2) if historial_jugador2 else 0

    mejor1 = max(historial_jugador1) if historial_jugador1 else 0
    mejor2 = max(historial_jugador2) if historial_jugador2 else 0

    total_dados = sum(conteo_caras.values()) # Suma total de todas las caras lanzadas.

    with stats_output: # Dirige la salida a la sección 'stats_output'.
        clear_output()
        print("ESTADÍSTICAS DEL JUEGO")
        print(f"Promedio Jugador 1: {promedio1:.2f}")
        print(f"Promedio Jugador 2: {promedio2:.2f}")
        print(f"Mejor turno Jugador 1: {mejor1}")
        print(f"Mejor turno Jugador 2: {mejor2}")
        print("\nFrecuencia de caras:")

        # Muestra la frecuencia de cada cara del dado.
        for cara in range(1, 7):
            cantidad = conteo_caras.get(cara, 0)
            porcentaje = (cantidad / total_dados * 100) if total_dados > 0 else 0
            print(f"Cara {cara}: {cantidad} veces ({porcentaje:.2f}%)")


# Muestra las probabilidades reales de las combinaciones obtenidas durante el juego.
def mostrar_probabilidades_reales():
    with probabilidades_output: # Dirige la salida a la sección 'probabilidades_output'.
        clear_output()

        print("PROBABILIDADES REALES GENERADAS POR EL JUEGO")
        print("Se actualizan con cada lanzamiento real de los dados.")
        print(f"Total de muestras tomadas: {total_muestras_combinaciones}")
        print("-" * 75)
        print(f"{'Combinación':30} {'Veces':12} {'Probabilidad real':18}")
        print("-" * 75)

        # Itera sobre las combinaciones estadísticas y muestra su frecuencia y probabilidad.
        for combinacion in combinaciones_estadisticas:
            cantidad = conteo_combinaciones.get(combinacion, 0)
            porcentaje = (cantidad / total_muestras_combinaciones * 100) if total_muestras_combinaciones > 0 else 0

            print(f"{combinacion:30} {cantidad:<12} {porcentaje:>10.2f}%")

        print("-" * 75)


# Reinicia el estado de los dados y los tiros para un nuevo turno.
def reiniciar_turno():
    global tiros_restantes, dados_actuales # Accede a las variables globales.

    tiros_restantes = 3 # Restablece los tiros disponibles.
    dados_actuales = [1, 1, 1, 1, 1] # Reinicia los dados a su estado inicial.

    # Desbloquea y desmarca todos los checkboxes de los dados.
    for checkbox in dice_checkboxes:
        checkbox.value = False
        checkbox.disabled = False

    actualizar_dados() # Actualiza la visualización de los dados.

    roll_button.disabled = False # Habilita el botón de lanzar dados.
    save_button.disabled = True # Deshabilita el botón de guardar puntaje.

    mostrar_turno() # Actualiza la información del turno.

    with dice_output:
        clear_output()
        print("Nuevo turno. Lanza los dados.")


# ===============================
# Funciones principales del juego (lógica)
# ===============================
# Maneja el evento de click del botón "Lanzar dados".
def lanzar_dados(b):
    global tiros_restantes, dados_actuales # Accede a variables globales.
    global total_muestras_combinaciones

    if tiros_restantes > 0: # Solo permite lanzar si quedan tiros.

        for i in range(5):
            if not dice_checkboxes[i].value: # Si el dado NO está bloqueado.
                dados_actuales[i] = random.randint(1, 6) # Lanza el dado.

        conteo_caras.update(dados_actuales) # Actualiza el conteo global de caras.

        combinacion_actual = calcular_combinacion_actual(dados_actuales) # Clasifica la combinación obtenida.
        conteo_combinaciones[combinacion_actual] += 1 # Actualiza el conteo de combinaciones.
        total_muestras_combinaciones += 1 # Incrementa el contador de muestras.

        tiros_restantes -= 1 # Decrementa los tiros restantes.
        actualizar_dados() # Actualiza la visualización de los dados.

        # Determina la mejor categoría posible y su puntaje para mostrar al jugador.
        categoria, puntaje_posible = mejor_categoria(dados_actuales, jugador_actual)

        with dice_output: # Muestra la información del lanzamiento.
            clear_output()
            print(f"Dados actuales: {dados_actuales}")
            print(f"Repeticiones en este tiro: {dict(Counter(dados_actuales))}")
            print(f"Combinación obtenida: {combinacion_actual}")
            print(f"Mejor categoría automática: {categoria}")
            print(f"Puntaje posible: {puntaje_posible}")

        mostrar_turno() # Actualiza la información del turno.
        mostrar_estadisticas() # Actualiza las estadísticas.
        mostrar_probabilidades_reales() # Actualiza las probabilidades reales.

        save_button.disabled = False # Habilita el botón de guardar puntaje.

        if tiros_restantes == 0: # Si no quedan más tiros.
            roll_button.disabled = True # Deshabilita el botón de lanzar.
            with result_output:
                clear_output()
                print("Ya no quedan tiros. Guarda el puntaje.")


# Maneja el evento de click del botón "Guardar puntaje".
def guardar_puntaje(b):
    global jugador_actual, ronda_general # Accede a variables globales.

    # Obtiene la mejor categoría y puntaje para los dados actuales del jugador.
    categoria, puntaje = mejor_categoria(dados_actuales, jugador_actual)

    if categoria is None: # Si no hay categorías disponibles para puntuar (debería ser raro en juego normal).
        finalizar_juego() # Finaliza el juego.
        return

    puntajes[jugador_actual][categoria] = puntaje # Guarda el puntaje en la categoría correspondiente.

    # Registra el puntaje en el historial del jugador y cambia al siguiente jugador.
    if jugador_actual == 1:
        historial_jugador1.append(puntaje)
        jugador_actual = 2
    else:
        historial_jugador2.append(puntaje)
        jugador_actual = 1
        ronda_general += 1 # Incrementa la ronda general cuando ambos jugadores han terminado su turno.

    # Calcula los totales actuales para los gráficos.
    total1 = total_jugador(1) + calcular_prima(1)
    total2 = total_jugador(2) + calcular_prima(2)

    turnos.append(len(turnos) + 1) # Añade el número de turno al registro.
    puntos_j1_grafica.append(total1) # Guarda el puntaje acumulado J1.
    puntos_j2_grafica.append(total2) # Guarda el puntaje acumulado J2.

    with result_output: # Muestra un mensaje de confirmación.
        clear_output()
        print(f"Se guardó automáticamente en: {categoria}")
        print(f"Puntaje guardado: {puntaje}")

    mostrar_tabla() # Actualiza la tabla de puntuación.
    mostrar_estadisticas() # Actualiza las estadísticas.
    mostrar_probabilidades_reales() # Actualiza las probabilidades reales.

    if ronda_general > max_rondas: # Si todas las rondas han terminado.
        finalizar_juego() # Finaliza el juego.
    else:
        reiniciar_turno() # Reinicia para el siguiente turno.


# Maneja el evento de click del botón "Mostrar gráficos".
def mostrar_graficos(b):
    with grafico_output: # Dirige la salida a la sección 'grafico_output'.
        clear_output(wait=True) # Limpia la salida anterior.

        if len(turnos) == 0:
            print("Aún no hay datos para graficar.")
            return

        # Gráfico de evolución de puntajes acumulados.
        plt.figure(figsize=(7, 4))
        plt.plot(turnos, puntos_j1_grafica, marker="o", label="Jugador 1")
        plt.plot(turnos, puntos_j2_grafica, marker="o", label="Jugador 2")
        plt.title("Evolución de puntajes acumulados")
        plt.xlabel("Turno")
        plt.ylabel("Puntaje acumulado")
        plt.legend()
        plt.grid(True)
        plt.show()

        # Realiza y grafica la simulación Montecarlo.
        resultados, caras_simuladas = simulacion_montecarlo(1000)

        plt.figure(figsize=(7, 4))
        plt.hist(resultados, bins=10)
        plt.title("Distribución de puntajes - Simulación Montecarlo")
        plt.xlabel("Puntaje")
        plt.ylabel("Frecuencia")
        plt.grid(True)
        plt.show()

        # Gráfico de frecuencia de caras de los dados lanzados durante el juego.
        caras = list(range(1, 7))
        frecuencias = [conteo_caras.get(cara, 0) for cara in caras]

        plt.figure(figsize=(7, 4))
        plt.bar(caras, frecuencias)
        plt.title("Frecuencia de caras durante el juego")
        plt.xlabel("Cara del dado")
        plt.ylabel("Cantidad de veces")
        plt.grid(axis="y")
        plt.show()

        # Gráfico de barras de las probabilidades reales de combinaciones.
        nombres = []
        porcentajes = []

        for combinacion in combinaciones_estadisticas:
            cantidad = conteo_combinaciones.get(combinacion, 0)
            porcentaje = (cantidad / total_muestras_combinaciones * 100) if total_muestras_combinaciones > 0 else 0
            nombres.append(combinacion)
            porcentajes.append(porcentaje)

        plt.figure(figsize=(10, 5))
        plt.bar(nombres, porcentajes)
        plt.title("Probabilidades reales de combinaciones durante el juego")
        plt.xlabel("Combinación")
        plt.ylabel("Probabilidad real (%)")
        plt.xticks(rotation=45, ha="right") # Rota las etiquetas del eje x para mejor legibilidad.
        plt.grid(axis="y")
        plt.show()

        # Gráfico de comparación de puntajes totales al final del juego.
        plt.figure(figsize=(6, 4))
        plt.bar(
            ["Jugador 1", "Jugador 2"],
            [total_jugador(1) + calcular_prima(1), total_jugador(2) + calcular_prima(2)] # Usa los puntajes finales.
        )
        plt.title("Comparación de puntajes totales")
        plt.ylabel("Puntos")
        plt.grid(axis="y")
        plt.show()


# Finaliza el juego, deshabilita los controles y muestra al ganador.
def finalizar_juego():
    roll_button.disabled = True # Deshabilita el botón de lanzar.
    save_button.disabled = True # Deshabilita el botón de guardar.

    # Deshabilita todos los checkboxes de los dados.
    for checkbox in dice_checkboxes:
        checkbox.disabled = True

    # Calcula los puntajes finales de ambos jugadores.
    total1 = total_jugador(1) + calcular_prima(1)
    total2 = total_jugador(2) + calcular_prima(2)

    # Determina el ganador o si hay un empate.
    if total1 > total2:
        ganador = "Ganó el Jugador 1"
    elif total2 > total1:
        ganador = "Ganó el Jugador 2"
    else:
        ganador = "Empate"

    with turn_output:
        clear_output()
        print("JUEGO TERMINADO")

    with result_output:
        clear_output()
        print(ganador)
        print(f"Puntaje final Jugador 1: {total1}")
        print(f"Puntaje final Jugador 2: {total2}")
        print("Puedes presionar el botón de gráficos para ver los resultados.")


# ===============================
# Conectar botones a sus funciones
# ===============================
roll_button.on_click(lanzar_dados) # Asocia la función lanzar_dados al evento de click del botón.
save_button.on_click(guardar_puntaje) # Asocia la función guardar_puntaje al evento de click del botón.
grafico_button.on_click(mostrar_graficos) # Asocia la función mostrar_graficos al evento de click del botón.

# ===============================
# Mostrar interfaz inicial
# ===============================
# Muestra todos los widgets en el orden deseado en la salida de la celda.
display(
    turn_output, # Información del turno.
    dice_container, # Dados y checkboxes.
    widgets.HBox([roll_button, save_button, grafico_button]), # Botones agrupados horizontalmente.
    dice_output, # Salida de los dados y mejor categoría.
    result_output, # Mensajes de resultado.
    score_output, # Tabla de puntuación.
    stats_output, # Estadísticas.
    probabilidades_output, # Probabilidades reales.
    grafico_output # Gráficos.
)

# Llama a las funciones para mostrar la información inicial al cargar el juego.
mostrar_turno()
mostrar_tabla()
mostrar_estadisticas()
mostrar_probabilidades_reales()

Output()

Output()

Output()

Output()

Output()

Output()

Output()